# Recall Bench v2 — wide field
25 frozen questions and an answer key (`data/`, `questions/`). All v2 output goes under `v2/`.

**Bring your own data.** No data ships with this notebook. Before cell 0, put these three files under `./local-data/`:
- `data/recalls_contestant.csv`: the records the contestants answer from (FDA food enforcement records, one row per
  record, with `event_id`, `recalling_firm`, `state`, `classification`, `reason_for_recall`, `recall_initiation_date`
  and `report_date` columns).
- `data/answer_key.json`: values computed from that CSV with pandas; the judge prompt reads `n_records`, `n_events`,
  `by_classification`, `reason_cat_records`, `reason_cat_events`, `top_firms_records`, `top_states_records`, `non_US`,
  `median_lag_days_init_to_report` and `max_lag_days`.
- `questions/questions.json`: `{"word_budget": <int>, "questions": [{"id", "type", "question", "rubric",
  "reference", "judge_facts"}]}`. `gpt-eval-9426.ipynb` (cells 2 and 2b) builds this file from the CSV and
  the key.

Cell 0 prints `MISSING` for any of the three it cannot find.

The deterministic grader (the cell headed `2. deterministic grader`, notebook cell index 4) hard-codes the original
question ids q01 to q15: their units (`event` for q02, q09 and q12, `record` for the rest), the totals a count
must sum to (q04, q05, q06 and q11 to 100, q12 to 52), a named firm for q09 and a wider tolerance for q14.
`type` is `deterministic` (graded by that cell, and rep 1 also by the judges) or `interpretive` (judges only).
Rewrite that cell for your own questions.
The judge prompt (the cell headed `3. two-judge pass`, notebook cell index 5) says "a CSV of 100 FDA food
enforcement records", and the runner's `SYSTEM` and `RULES` text (the cell headed `1. grid runner`, notebook cell index
3) describe the original CSV and its reason categories; rewrite them too.

Contestants run via OpenRouter; judges are GPT-6 Astra (OpenAI direct) and Gemini 3.1 Pro (OpenRouter).
Run cells top to bottom. Every cell is self-contained and resumable.

In [ ]:
# ── 0. setup + smoke test ─────────────────────────────────────────
import os, json, time, pathlib
from openai import OpenAI

VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"
for d in ["runs","judged","results"]: (V2/d).mkdir(parents=True, exist_ok=True)

oai = OpenAI(timeout=180, max_retries=0)  # max_retries=0: SDK retries would be unreserved paid requests
orx = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", timeout=180, max_retries=0)

FIELD = [  # (slug, label, [endpoint tags in order] or None = OpenRouter credit, any endpoint)
    ("anthropic/claude-fable-5.1",          "fable-5.1",        ["amazon-bedrock"]),                 # untagged = cross-region, list price; needs account retention aws_review
    ("anthropic/claude-opus-5",             "opus-5",           ["amazon-bedrock"]),
    ("anthropic/claude-sonnet-5",           "sonnet-5",         ["amazon-bedrock/global"]),          # no untagged endpoint for Sonnet 5
    ("google/gemini-3.8-flash",             "gemini-3.8-flash", ["google-vertex/global"]),
    ("x-ai/grok-4.6",                       "grok-4.6",         ["amazon-bedrock/us-west-2"]),       # only Bedrock endpoint; regional tag required
    ("moonshotai/kimi-k3",                  "kimi-k3",          ["baseten/fp8","digitalocean"]),
    ("z-ai/glm-5.3",                        "glm-5.3",          ["baseten/fp4","digitalocean" ]),    # DO first; baseten is fp4 (logged per call)
    ("deepseek/deepseek-v4-pro-0813",       "deepseek-v4-pro",  ["digitalocean"]),
    ("openai/gpt-5.6-sol",                  "sol-via-or",       ["openai"]),
    ("nvidia/nemotron-3-ultra-550b-a55b",   "nemotron-3-ultra", ["baseten/fp4"]),
    ("qwen/qwen3.8-max",                    "qwen3.8-max",      None),   # alibaba only; OR credit
    ("qwen/qwen3.8-flash",                  "qwen3.8-flash",    None),
    ("meta/muse-spark-1.3",                 "muse-spark-1.3",   None),   # meta only; OR credit
    ("minimax/minimax-m3",                  "minimax-m3",       ["minimax/fp8"]),                    # OR credit, but pinned so quant is fixed
]
def pin(tags):
    return {"provider": {"order": tags, "allow_fallbacks": False}} if tags else {}

# ── resolver: validate every tag against the live endpoints API ──────
import requests
# --- Spend guard -----------------------------------------------------------
# Every paid call (OpenRouter through the SDK, direct OpenAI, direct Bedrock) is reserved at its worst case before it
# is sent and settled after (guarded_chat, guarded_responses, guarded_converse). A call is refused, and the run stops,
# if its model has no entry in PRICE, if its body has no max_tokens, or if its worst case would take the run past
# RUN_BUDGET_USD. PRICE holds each model's listed price in USD per million tokens (input, output). A model whose
# price is not confirmed is left out of PRICE, and the notebook then refuses to run (checked below, before any paid call).
from datetime import datetime, timezone
import math
import os

# --- Canonical env parsing and ledger-root rule (identical in every notebook) ---------------------------------
# RUN_BUDGET_USD (this notebook reads no other amount): unset means the default. Any value that is set, a blank or whitespace
# value included, must parse as a finite float greater than 0, or the notebook refuses to start.
def _env_usd(name, default):
    raw = os.environ.get(name)
    if raw is None:
        return float(default)
    try:
        v = float(raw.strip())
    except ValueError:
        v = float("nan")
    if not math.isfinite(v) or v <= 0:
        raise RuntimeError(f"Refusing to start: {name} must be a finite number greater than 0 (got {raw!r}).")
    return v


# Ledger root. If SPEND_LEDGER_DIR is set at all, it is stripped: blank or whitespace-only refuses, anything else
# becomes os.path.abspath(os.path.expanduser(value)). If it is not set, the root is <repo root>/spend_ledger, where
# the repo root is the first directory, starting at the absolute working directory and walking up, for which
# os.path.exists(<dir>/.git) is true (a .git directory or a .git file). With no such directory the notebook refuses:
# there is no fallback to the working directory, because a different working directory would give a different lock.
# The lock is <root>/runner.lock and the run ledgers are <root>/<RUN_ID>/spend_ledger.jsonl.
def _spend_root():
    raw = os.environ.get("SPEND_LEDGER_DIR")
    if raw is not None:
        value = raw.strip()
        if not value:
            raise RuntimeError("Refusing to start: SPEND_LEDGER_DIR is set but blank. Unset it to use "
                               "<repo root>/spend_ledger, or set it to a directory.")
        return os.path.abspath(os.path.expanduser(value))
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, ".git")):
            return os.path.join(d, "spend_ledger")
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError(f"Refusing to start: SPEND_LEDGER_DIR is not set and there is no .git above the working "
                               f"directory ({start}). Set SPEND_LEDGER_DIR to the shared ledger directory.")
        d = parent


RUN_BUDGET_USD = _env_usd("RUN_BUDGET_USD", 25.00)   # cap per run in USD; set the RUN_BUDGET_USD environment variable to change it
RUN_ID = None              # None = new run with a fresh ledger. Set a non-empty string to resume and keep its spend.
# Confirmed listed prices, USD per million tokens (input, output), retrieved 2026-11-05. Each is the highest listed
# rate that could apply to a plain request (priority or regional endpoints, long-context bands), so reservations are
# conservative. Sources: https://openrouter.ai/api/v1/models/<slug>/endpoints and the model pages at
# https://openrouter.ai/<slug> for the OpenRouter models; https://developers.openai.com/api/docs/pricing for gpt-6-astra.
PRICE = {
    "anthropic/claude-fable-5.1":        (10.0, 50.0),
    "anthropic/claude-opus-5":           (10.0, 50.0),
    "anthropic/claude-sonnet-5":         (2.2, 11.0),
    "google/gemini-3.8-flash":           (1.35, 6.75),
    "x-ai/grok-4.6":                     (8.0, 24.0),
    "moonshotai/kimi-k3":                (4.5, 22.5),
    "z-ai/glm-5.3":                      (2.8, 8.8),
    "deepseek/deepseek-v4-pro-0813":     (1.98, 8.0),
    "openai/gpt-5.6-sol":                (8.8, 33.0),
    "nvidia/nemotron-3-ultra-550b-a55b": (0.625, 3.125),
    "qwen/qwen3.8-max":                  (2.0, 6.0),
    "qwen/qwen3.8-flash":                (0.15, 0.47),
    "meta/muse-spark-1.3":               (1.25, 4.25),
    "minimax/minimax-m3":                (0.6, 2.4),
    "google/gemini-3.1-pro-preview":     (7.2, 32.4),
    "gpt-6-astra":                       (20.0, 75.0),    # OpenAI direct, long-context standard rate
}
REQUIRED_MODELS = [_s for _s, _l, _t in FIELD] + ["google/gemini-3.1-pro-preview", "gpt-6-astra"]   # every model this notebook can call
missing_prices = sorted(set(REQUIRED_MODELS) - set(PRICE))
if missing_prices:
    raise RuntimeError(f"Not run: no confirmed price for {missing_prices}. Set each model's listed price in PRICE "
                       "(USD per million tokens, input and output) in this cell, then rerun.")
# None is the only signal for a new run. Any other value is a resume ID and goes through _checked_run_id below,
# which refuses an empty string, 0, False and every other non-string value. Nothing is defaulted.
if RUN_ID is None:
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
def _checked_run_id(value):
    """RUN_ID names one directory under SPEND_ROOT. Refuse an absolute path or any '..' segment, so the text of
    RUN_ID cannot lead os.path.join(SPEND_ROOT, RUN_ID) outside SPEND_ROOT. This is a string check only: it does
    not detect a symlink that already exists under SPEND_ROOT. Fails loudly."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(f"Refusing to start: RUN_ID must be a non-empty string (got {value!r}).")
    if os.path.isabs(value) or value.startswith(("/", "\\")) or os.path.splitdrive(value)[0]:
        raise RuntimeError(f"Refusing to start: RUN_ID must be a relative name, not an absolute path (got {value!r}).")
    if ".." in value.replace("\\", "/").split("/"):
        raise RuntimeError(f"Refusing to start: RUN_ID must not contain a '..' segment (got {value!r}).")
    return value
RUN_ID = _checked_run_id(RUN_ID)
# One ledger root and one lock for every notebook: SPEND_LEDGER_DIR, default <repo root>/spend_ledger (rule in _spend_root above).
SPEND_ROOT = _spend_root()
RUN_DIR = os.path.join(SPEND_ROOT, RUN_ID)
os.makedirs(RUN_DIR, exist_ok=True)
LEDGER_PATH = os.path.join(RUN_DIR, "spend_ledger.jsonl")

# One paid runner at a time across all notebooks. A single exclusive lock file in the shared ledger root: a second
# runner refuses to start while it exists. It is removed on a clean exit only. A stale lock (a runner that died
# without exiting) is never cleared automatically: confirm no runner is alive, then delete SPEND_ROOT/runner.lock by hand.
import atexit
RUNNER_LOCK = os.path.join(SPEND_ROOT, "runner.lock")
try:
    _lock_fd = os.open(RUNNER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
except FileExistsError:
    raise RuntimeError(f"{RUNNER_LOCK} exists. Another runner may be active, or a runner stopped without "
                       "releasing it. Refusing to start. Check for a live runner before deleting the lock by hand.")
_RUNNER_TOKEN = f"pid {os.getpid()} {datetime.now(timezone.utc).isoformat()}"
os.write(_lock_fd, _RUNNER_TOKEN.encode())
os.close(_lock_fd)


def _release_runner_lock():
    try:
        with open(RUNNER_LOCK) as f:
            if f.read() == _RUNNER_TOKEN:
                os.remove(RUNNER_LOCK)
    except OSError:
        pass


atexit.register(_release_runner_lock)


import json
import threading, uuid
from decimal import Decimal


class BudgetExceeded(RuntimeError):
    pass


PROMPT_OVERHEAD_TOKENS = 256    # allowance for chat-template tokens that the byte bound below does not cover


def max_price_for(model):
    """provider.max_price for one model: its listed price in USD per million tokens (input, output). PRICE already
    holds per-million figures, so the value is converted with Decimal and not scaled. An unpriced model is refused."""
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    pin, pout = PRICE[model]
    return {"prompt": float(Decimal(str(pin))), "completion": float(Decimal(str(pout)))}


def openrouter_body(body):
    """The body every OpenRouter call sends: the caller's body with provider.max_price set from PRICE and usage
    accounting switched on. Done here, once, so no call site can leave either out."""
    b = dict(body)
    b["provider"] = {**(body.get("provider") or {}), "max_price": max_price_for(body["model"])}
    b["usage"] = {"include": True}
    return b


def worst_case_cost(body):
    """Upper bound for one attempt. Input comes from the UTF-8 byte count of the messages (each token covers
    at least one byte); output is capped by max_tokens, which includes reasoning. Fails closed."""
    model = body["model"]
    if model not in PRICE:
        raise BudgetExceeded(f"not sent: no price for {model}. Add it to PRICE in the config cell.")
    if not isinstance(body.get("max_tokens"), int):
        raise BudgetExceeded(f"not sent: request for {model} has no max_tokens, so its cost cannot be bounded.")
    pin, pout = PRICE[model]
    in_bound = len(json.dumps(body["messages"], ensure_ascii=False).encode("utf-8")) + PROMPT_OVERHEAD_TOKENS
    return pin / 1e6 * in_bound + pout / 1e6 * body["max_tokens"]


def _is_count(x):
    return isinstance(x, int) and not isinstance(x, bool) and x >= 0


def attempt_cost(body, usage):
    """Settle only against complete, validated provider usage: a reported cost that is finite and above zero,
    and both token counts as non-negative integers. Anything less keeps the worst case that was reserved, so a
    zero, missing, partial or malformed reply cannot release the reservation."""
    if isinstance(usage, dict):
        c = usage.get("cost")
        if (isinstance(c, (int, float)) and not isinstance(c, bool) and 0 < c < float("inf")
                and _is_count(usage.get("prompt_tokens")) and _is_count(usage.get("completion_tokens"))):
            return float(c), "usage"
    return worst_case_cost(body), "worst_case"


_ledger_lock = threading.Lock()
_ledger = {}                    # attempt id -> entry; reloaded from LEDGER_PATH on resume
_budget_halted = False          # set by a refused dispatch or by an overrun; no later attempt is sent
if os.path.exists(LEDGER_PATH):
    with open(LEDGER_PATH) as _f:
        for _line in _f:
            if _line.strip():
                _rec = json.loads(_line)
                _ledger.setdefault(_rec["id"], {}).update(_rec)
    _budget_halted = any(_e.get("overrun") for _e in _ledger.values())   # an overrun halts the run after a restart too


def ledger_total():             # caller holds _ledger_lock. A reserved attempt with no settled line counts at worst case.
    return sum(e["cost"] if e["state"] == "settled" else e["worst_case"] for e in _ledger.values())


def _ledger_write(rec):
    with open(LEDGER_PATH, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def ledger_reserve(body, purpose):
    """Reserve one attempt at its worst case before it is sent. Raises BudgetExceeded if the run is halted, or if
    the reservation would take the run past RUN_BUDGET_USD; landing exactly on the cap is allowed."""
    global _budget_halted
    wc = round(worst_case_cost(body), 6)
    with _ledger_lock:
        if _budget_halted:
            raise BudgetExceeded(f"not sent: the run is halted, so {body['model']} ({purpose}) is refused before dispatch")
        if ledger_total() + wc <= RUN_BUDGET_USD:
            aid = uuid.uuid4().hex
            e = {"id": aid, "state": "reserved", "model": body["model"], "purpose": purpose,
                 "worst_case": wc, "ts": datetime.now(timezone.utc).isoformat()}
            _ledger[aid] = e
            _ledger_write(e)
            return aid
        _budget_halted = True
    raise BudgetExceeded(f"not sent: {body['model']} ({purpose}), worst case ${wc:.4f}, "
                         f"would take the run past RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}")


def ledger_settle(aid, body, usage):
    """Settle one attempt. A reported charge above the reservation for this attempt is an overrun: it is written to
    the ledger with its reservation and halts the run, so every later dispatch is refused before any request is sent."""
    global _budget_halted
    cost, source = attempt_cost(body, usage)
    with _ledger_lock:
        e = _ledger[aid]
        charged = round(cost, 6)
        overrun = source == "usage" and charged > e["worst_case"]
        e.update(state="settled", cost=charged, cost_source=source)
        rec = {"id": aid, "state": "settled", "cost": charged, "cost_source": source}
        if overrun:
            e["overrun"] = True
            rec.update(overrun=True, reserved=e["worst_case"])
            _budget_halted = True
        _ledger_write(rec)
    if overrun:
        print(f"Ledger overrun: an attempt was charged ${charged:.4f} against a reservation of ${e['worst_case']:.4f}. "
              f"The run is halted; no later attempt will be sent.")
    return e["cost"]


def ledger_summary():
    with _ledger_lock:
        es, total = list(_ledger.values()), ledger_total()
    by = {}
    for e in es:
        n, c = by.get(e["purpose"], (0, 0.0))
        by[e["purpose"]] = (n + 1, c + (e["cost"] if e["state"] == "settled" else e["worst_case"]))
    lines = [f"Spend ledger: {len(es)} attempts, RUN_BUDGET_USD = ${RUN_BUDGET_USD:.2f}"]
    lines += [f"  {p:8s} attempts={n:5d}  charged=${c:.4f}" for p, (n, c) in sorted(by.items())]
    lines.append(f"  total charged ${total:.4f}   reserved, not settled={sum(e['state'] == 'reserved' for e in es)}")
    return "\n".join(lines)


def _sdk_text(kw):
    """The request text, as a bound on input size (every field except the model and the output cap)."""
    return json.dumps({k: v for k, v in kw.items() if k not in ("model", "modelId", "extra_body")}, default=str,
                      ensure_ascii=False)


def guarded_chat(client, purpose, **kw):
    """OpenRouter chat call through the SDK. provider.max_price and usage include go in extra_body; the attempt is
    reserved at its worst case first and settled to the cost OpenRouter reports."""
    body = openrouter_body({"model": kw["model"], "messages": kw["messages"], "max_tokens": kw.get("max_tokens")})
    eb = dict(kw.get("extra_body") or {})
    eb["provider"] = {**(eb.get("provider") or {}), "max_price": body["provider"]["max_price"]}
    eb["usage"] = {"include": True}
    kw = {**kw, "extra_body": eb}
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = client.chat.completions.create(**kw)
        try:
            usage = r.usage.model_dump() if r.usage is not None else None
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def _usage_charge(model, tokens_in, tokens_out):
    """Direct calls report token counts, not cost. Charge them at the listed PRICE only when both counts are complete;
    otherwise return None so the worst case that was reserved stays."""
    if model in PRICE and _is_count(tokens_in) and _is_count(tokens_out):
        pin, pout = PRICE[model]
        c = pin / 1e6 * tokens_in + pout / 1e6 * tokens_out
        if 0 < c < float("inf"):
            return {"cost": c, "prompt_tokens": tokens_in, "completion_tokens": tokens_out}
    return None


def guarded_responses(client, purpose, **kw):
    """Direct OpenAI Responses call. No OpenRouter fields apply. The attempt is reserved first at listed price times
    the input bound plus listed price times max_output_tokens, then settled on the reported token usage when it is
    complete, else at the worst case. PRICE must list the model."""
    body = {"model": kw["model"], "messages": [{"role": "user", "content": _sdk_text(kw)}],
            "max_tokens": kw.get("max_output_tokens")}
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = client.responses.create(**kw)
        try:
            u = r.usage
            usage = _usage_charge(kw["model"], getattr(u, "input_tokens", None), getattr(u, "output_tokens", None))
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)


def guarded_converse(client, purpose, **kw):
    """Direct Bedrock Converse call. No OpenRouter fields apply. Reserved at the worst case first, settled on the
    reported token usage when it is complete, else at the worst case. inferenceConfig.maxTokens bounds the output;
    PRICE must list the model id."""
    mt = (kw.get("inferenceConfig") or {}).get("maxTokens")
    body = {"model": kw["modelId"], "messages": [{"role": "user", "content": _sdk_text(kw)}], "max_tokens": mt}
    aid = ledger_reserve(body, purpose)
    usage = None
    try:
        r = client.converse(**kw)
        try:
            u = r["usage"]
            usage = _usage_charge(kw["modelId"], u.get("inputTokens"), u.get("outputTokens"))
        except Exception:
            usage = None
        return r
    finally:
        ledger_settle(aid, body, usage)



def endpoints(slug):
    r = requests.get(f"https://openrouter.ai/api/v1/models/{slug}/endpoints", timeout=30)
    r.raise_for_status(); return r.json()["data"]["endpoints"]
print("endpoint check:")
for slug, label, tags in FIELD:
    eps = {e["tag"]: e for e in endpoints(slug)}
    if tags is None:
        cheapest = min(eps.values(), key=lambda e: float(e["pricing"]["prompt"]))
        print(f"  {label:18s} OR-credit  any of {len(eps)} endpoints; cheapest {cheapest['tag']} ${float(cheapest['pricing']['prompt'])*1e6:.2f}/{float(cheapest['pricing']['completion'])*1e6:.2f}")
        continue
    for t in tags:
        if t in eps:
            e = eps[t]
            print(f"  {label:18s} {t:28s} ${float(e['pricing']['prompt'])*1e6:.2f}/{float(e['pricing']['completion'])*1e6:.2f}  quant={e.get('quantization')}")
        else:
            print(f"  {label:18s} {t:28s} MISSING — available: {sorted(eps)}")
EFFORTS = ["medium", "high"]
JUDGES = {"astra": ("oai", "gpt-6-astra"), "gemini": ("orx", "google/gemini-3.1-pro-preview")}

# smoke: one tiny call per contestant + judge; prints provider OR actually used
ping = "Reply with the single word: ready"
for slug, label, tags in FIELD:
    try:
        r = guarded_chat(orx, "call", model=slug, max_tokens=300, messages=[{"role":"user","content":ping}], extra_body={"reasoning":{"effort":"low"}, **pin(tags)})
        got = getattr(r, "provider", None) or (r.model_extra or {}).get("provider")
        print(f"[{label:18s}] ok  pinned={(tags[0] if tags else 'OR-credit'):28s} got={got}  {(r.choices[0].message.content or '').strip()[:20]!r}")
    except Exception as e:
        print(f"[{label:18s}] FAILED: {str(e)[:120]}")
for name, (client, mid) in JUDGES.items():
    try:
        if client == "oai":
            r = guarded_responses(oai, "call", model=mid, input=ping, max_output_tokens=30, reasoning={"effort":"low"}, service_tier="default")
            print(f"[judge {name}] ok {r.model}")
        else:
            r = guarded_chat(orx, "call", model=mid, max_tokens=30, messages=[{"role":"user","content":ping}])
            print(f"[judge {name}] ok {r.model}")
    except Exception as e:
        print(f"[judge {name}] FAILED: {str(e)[:120]}")
for f in ["data/recalls_contestant.csv","data/answer_key.json","questions/questions.json"]:
    print(f, (VOL/f).stat().st_size if (VOL/f).exists() else "MISSING")


In [ ]:
import pathlib
RUNS = pathlib.Path("./local-data/v2/runs")
n = len([p.unlink() for p in RUNS.glob("kimi-k3__*.json")])
print("removed", n, "kimi runs (DO quant unknown, Baseten is fp8)")

In [ ]:
# ── 1. grid runner (resumable, one JSON per call) ─────────────────
import json, pathlib, time, random
VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"; RUNS = V2/"runs"
qdoc = json.loads((VOL/"questions"/"questions.json").read_text())
QS = qdoc["questions"]; BUDGET = qdoc["word_budget"]
CSV = (VOL/"data"/"recalls_contestant.csv").read_text()
REPS = 3
ONLY = None            # e.g. ["fable-5.1"] to run a subset by label
LABEL = {s:l for s,l,_ in FIELD}; PROV = {l:(t[0] if t else None) for _,l,t in FIELD}; TAGS = {l:t for _,l,t in FIELD}

RULES = ("Reason categories, when asked: categorize on reason_for_recall only, case-insensitive, "
 "first match wins in this order: listeria='listeria'; salmonella='salmonella'; "
 "ecoli='e. coli'/'e.coli'/'stec'; other_micro=hepatitis/norovirus/clostridium/cronobacter/mold/yeast/botul; "
 "undeclared_allergen='undeclared'/'allergen'; foreign_material=foreign/metal/plastic/glass/wood/rubber; else other.")
SYSTEM = f"""You are analyzing a CSV of FDA food enforcement records. Answer only from the CSV. No outside knowledge, no tools.
Answer in at most {BUDGET} words. Anything past {BUDGET} words is discarded before grading. State the unit you used (records or events) whenever counts are involved.
{RULES}"""

def run_id(label, effort, qid, rep): return f"{label}__{effort}__{qid}__r{rep}"

def call(slug, label, effort, q, rep):
    user = f"<csv>\n{CSV}\n</csv>\n\n<question id=\"{q['id']}\">\n{q['question']}\n</question>"
    err = "unknown"
    for attempt in range(6):
        t0 = time.time()
        try:
            r = guarded_chat(orx, "call", model=slug, max_tokens=4000, messages=[{"role":"system","content":SYSTEM},{"role":"user","content":user}], extra_body={"reasoning":{"effort":effort}, "usage":{"include":True}, **pin(TAGS[label])})
            u = r.usage; ud = getattr(u, "model_extra", {}) or {}
            ptd = getattr(u, "prompt_tokens_details", None); ctd = getattr(u, "completion_tokens_details", None)
            return {"run_id": run_id(label,effort,q["id"],rep), "model_req": slug, "label": label, "pinned": PROV[label],
                "model_resolved": r.model, "provider": ud.get("provider") or getattr(r,"provider",None) or (r.model_extra or {}).get("provider"),
                "effort": effort, "qid": q["id"], "rep": rep, "response_id": r.id,
                "answer": r.choices[0].message.content or "", "finish": r.choices[0].finish_reason,
                "in_tok": u.prompt_tokens, "out_tok": u.completion_tokens,
                "cached_tok": getattr(ptd, "cached_tokens", None) if ptd else None,
                "reason_tok": getattr(ctd, "reasoning_tokens", None) if ctd else None,
                "or_cost": ud.get("cost"), "wall_s": round(time.time()-t0,1), "retries": attempt,
                "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}
        except Exception as e:
            err = str(e)[:300]
            if any(k in err for k in ["429","502","503","504","timeout","Timeout","overloaded"]):
                wait = min(90, 2**attempt + random.random()); print(f"  retry {attempt} in {wait:.0f}s: {err[:70]}"); time.sleep(wait)
            else:
                break
    return {"run_id": run_id(label,effort,q["id"],rep), "model_req": slug, "label": label, "effort": effort,
            "qid": q["id"], "rep": rep, "error": err, "ts": time.strftime("%Y-%m-%dT%H:%M:%S")}

done = {p.stem for p in RUNS.glob("*.json")}
for slug, label, _p in FIELD:
    if ONLY and label not in ONLY: continue
    for effort in EFFORTS:
        todo = [(q,rep) for rep in range(1,REPS+1) for q in QS if run_id(label,effort,q["id"],rep) not in done]
        if not todo: continue
        print(f"\n== {label} @ {effort}: {len(todo)} calls")
        tin=tout=tr=tc=0; cost=0.0; t0=time.time(); errs=0
        for i,(q,rep) in enumerate(todo,1):
            rec = call(slug, label, effort, q, rep)
            (RUNS/f"{rec['run_id']}.json").write_text(json.dumps(rec, indent=1))
            if "error" in rec:
                errs += 1; print(f"  ERROR {rec['run_id']}: {rec['error'][:90]}")
                if errs >= 5 and i <= 6: print("  aborting this config; fix and rerun"); break
                continue
            tin+=rec["in_tok"]; tout+=rec["out_tok"]; tr+=rec["reason_tok"] or 0; tc+=rec["cached_tok"] or 0; cost+=rec["or_cost"] or 0
            if i % 15 == 0 or i == len(todo):
                print(f"  {i}/{len(todo)} in={tin:,} cached={tc:,} out={tout:,} reason={tr:,} or_cost=${cost:.2f} provider={rec['provider']} {time.time()-t0:.0f}s")
print("\ngrid complete:", len(list(RUNS.glob("*.json"))), "run files")


In [ ]:
# ── 2. deterministic grader (q01–q15), no model calls ─────────────
import json, re, pathlib, pandas as pd
VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"
qdoc = json.loads((VOL/"questions"/"questions.json").read_text())
QS = {q["id"]: q for q in qdoc["questions"]}; BUDGET = qdoc["word_budget"]
DET = {q["id"] for q in qdoc["questions"] if q["type"]=="deterministic"}

def truncate(t): return " ".join(t.split()[:BUDGET])
def norm(t): return " ".join(re.sub(r"[^a-z0-9 ]", " ", t.lower()).split())
def nums(t):
    t = re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\1\2\3", t)
    return [float(x.replace(",","")) for x in re.findall(r"\d[\d,]*\.?\d*", t)]
UNIT  = {qid: "event" if qid in ("q02","q09","q12") else "record" for qid in QS}
TOTAL = {"q04":100,"q05":100,"q06":100,"q11":100,"q12":52}
ALIAS = {"": ["blank","missing","empty","unknown","none","null","no state","nan"]}
def key_alias(k):
    if k in ALIAS: return ALIAS[k]
    toks = [t for t in norm(str(k)).split() if t not in ("llc","inc","co","ltd","the")]
    return [" ".join(toks[:2]) if len(toks) > 1 else toks[0]]
def value_near(text, key_, val, window=60):
    t = norm(text)
    for a in key_alias(key_):
        for m in re.finditer(re.escape(a), t):
            if any(abs(n-float(val)) < 0.5 for n in nums(t[m.end():m.end()+window])): return True
    return False
def grade(q, answer):
    a = truncate(answer); ref = q["reference"]; qid = q["id"]; crit = {}
    if isinstance(ref, (int, float)):
        crit["value"] = any(abs(n-ref) <= (1.0 if qid=="q14" else 0.5) for n in nums(a))
    elif isinstance(ref, list):
        crit["value"] = all(any(abs(n-v) < 0.5 for n in nums(a)) for v in ref)
    elif isinstance(ref, dict):
        if qid == "q09":
            t = norm(a); crit["value"] = "gellert" in t and value_near(a, "GELLERT GLOBAL GROUP", 2)
            crit["others_one"] = bool(re.search(r"\b(all|every|each|remaining|other|others)\b[^.]{0,60}\b(1|one)\b", t)) or bool(re.search(r"\b(1|one)\b[^.]{0,60}\b(all|every|each|remaining|other|others)\b", t))
        else:
            crit["value"] = all(value_near(a, k, v) for k, v in ref.items())
    crit["unit"] = UNIT[qid] in norm(a)
    if qid in TOTAL:
        c = nums(a); crit["sums"] = any(abs(sum(c[i:j])-TOTAL[qid]) < 0.5 for i in range(len(c)) for j in range(i+2, len(c)+1))
    crit["over_budget"] = len(answer.split()) > BUDGET
    crit["empty"] = len(answer.strip()) == 0
    return crit

rows = []
for p in sorted((V2/"runs").glob("*.json")):
    r = json.loads(p.read_text())
    if "error" in r or r["qid"] not in DET: continue
    c = grade(QS[r["qid"]], r["answer"]); sk = [k for k in c if k not in ("over_budget","empty")]
    rows.append({"run_id": r["run_id"], "label": r["label"], "effort": r["effort"], "qid": r["qid"], "rep": r["rep"],
                 **c, "score": sum(c[k] for k in sk)/len(sk), "reason_tok": r.get("reason_tok"), "answer": truncate(r["answer"])[:300]})
df = pd.DataFrame(rows); df.to_csv(V2/"judged"/"deterministic.csv", index=False)
print(f"{len(df)} graded answers\n")
piv = df.pivot_table(index=["label","effort"], columns="qid", values="value", aggfunc="mean").round(2)
piv["MEAN"] = df.groupby(["label","effort"]).value.mean().round(3)
print(piv.sort_values("MEAN", ascending=False).to_string())
fails = df[~df.value][["run_id","answer"]]
print(f"\n{len(fails)} value failures; first 15:")
for _, f in fails.head(15).iterrows(): print(f"- {f.run_id}\n    {f.answer[:200]}")


In [ ]:
# ── 3. two-judge pass: Astra (OpenAI direct) + Gemini 3.1 Pro (OR) ─
import os, json, re, pathlib, time, random
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
oai = OpenAI(timeout=180, max_retries=0)  # max_retries=0: SDK retries would be unreserved paid requests
orx = OpenAI(api_key=os.environ["OPENROUTER_API_KEY"], base_url="https://openrouter.ai/api/v1", timeout=180, max_retries=0)
VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"; JD = V2/"judged"
qdoc = json.loads((VOL/"questions"/"questions.json").read_text())
QS = {q["id"]: q for q in qdoc["questions"]}; BUDGET = qdoc["word_budget"]
KEY = json.loads((VOL/"data"/"answer_key.json").read_text())
JUDGES = {"astra": ("oai", "gpt-6-astra"), "gemini": ("orx", "google/gemini-3.1-pro-preview")}
JUDGE_DET_REP1 = True; WORKERS = 8
def truncate(t): return " ".join(t.split()[:BUDGET])
cleared = 0
for p in list(JD.glob("*__*.json")):
    if "error" in json.loads(p.read_text()): p.unlink(); cleared += 1
print(f"cleared {cleared} errored judge files")
KEYX = json.dumps({k: KEY[k] for k in ["n_records","n_events","by_classification","reason_cat_records","reason_cat_events",
        "top_firms_records","top_states_records","non_US","median_lag_days_init_to_report","max_lag_days"]})
def judge_prompt(q, answer):
    facts = q.get("judge_facts") or f"reference value: {json.dumps(q['reference'])}"
    crits = "\n".join(f"{i+1}. {c}" for i, c in enumerate(q["rubric"]))
    return f"""You are grading one answer to one question about a CSV of 100 FDA food enforcement records. Grade each rubric criterion PASS or FAIL. Nothing else counts: not length, not tone, not extra correct facts. A criterion passes only if the answer establishes it; silence is FAIL. If the answer states a wrong number for something the facts specify, the related criterion is FAIL.

<facts_for_grader>
{facts}
</facts_for_grader>

<answer_key_excerpt>
{KEYX}
</answer_key_excerpt>

<question>
{q["question"]}
</question>

<rubric>
{crits}
</rubric>

<answer>
{answer}
</answer>

Return JSON only, no prose, no fences:
{{"criteria":[{{"n":1,"pass":true|false,"why":"<10 words"}}, ...], "fabrication": true|false}}
"fabrication" is true only if the answer states a specific firm, number, or fact that contradicts the facts or the key."""
def parse(raw, q):
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    out = json.loads(raw)
    if len(out["criteria"]) != len(q["rubric"]): raise ValueError("criteria count mismatch")
    return out
def call_judge(jname, q, answer):
    client, mid = JUDGES[jname]; err = "unknown"
    for attempt in range(4):
        try:
            if client == "oai":
                r = guarded_responses(oai, "call", model=mid, input=judge_prompt(q, answer), reasoning={"effort":"medium"}, max_output_tokens=2000, service_tier="default")
                out = parse(r.output_text, q); model = r.model; tok = r.usage.total_tokens
            else:
                r = guarded_chat(orx, "call", model=mid, temperature=0, max_tokens=6000, messages=[{"role":"user","content":judge_prompt(q, answer)}], extra_body={"reasoning":{"effort":"low"}})
                out = parse(r.choices[0].message.content, q); model = r.model; tok = r.usage.total_tokens
            return {"judge": jname, "judge_model": model, "passes": [bool(c["pass"]) for c in out["criteria"]],
                    "why": [c.get("why","") for c in out["criteria"]], "fabrication": out.get("fabrication"), "judge_tok": tok}
        except Exception as e:
            err = str(e)[:200]; time.sleep(2**attempt + random.random())
    return {"judge": jname, "error": err}
def work(item):
    r, q, jname, out = item
    res = call_judge(jname, q, truncate(r["answer"]))
    res.update({"run_id": r["run_id"], "qid": r["qid"], "label": r["label"], "effort": r["effort"], "rep": r["rep"], "qtype": q["type"]})
    out.write_text(json.dumps(res, indent=1)); return res
runs = [json.loads(p.read_text()) for p in sorted((V2/"runs").glob("*.json"))]
todo = []
for r in runs:
    if "error" in r: continue
    q = QS[r["qid"]]
    if q["type"] == "interpretive" or (JUDGE_DET_REP1 and r["rep"] == 1):
        for jname in JUDGES:
            out = JD/f"{r['run_id']}__{jname}.json"
            if not out.exists(): todo.append((r, q, jname, out))
random.shuffle(todo); print(f"{len(todo)} judge calls to do, {WORKERS} workers")
tok = 0; fails = 0; done = 0; t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = [ex.submit(work, it) for it in todo]
    for f in as_completed(futs):
        res = f.result(); done += 1
        if "error" in res: fails += 1; print(f"  FAIL {res['run_id']} [{res['judge']}]: {res['error'][:100]}")
        tok += res.get("judge_tok", 0)
        if done % 50 == 0 or done == len(todo): print(f"  {done}/{len(todo)}  judge_tok={tok:,}  fails={fails}  {time.time()-t0:.0f}s")
print("judging complete")


In [ ]:
# ── 4. rollup → v2/results/summary.csv ─────────────────────────────
import json, pathlib, pandas as pd, numpy as np
VOL = pathlib.Path("./local-data"); V2 = VOL/"v2"
det = pd.read_csv(V2/"judged"/"deterministic.csv")
for c in ["value","unit","sums","others_one","over_budget","empty"]:
    if c in det: det[c] = det[c].astype(str).str.lower()=="true"
rows = []
for p in (V2/"judged").glob("*__*.json"):
    j = json.loads(p.read_text())
    if "error" in j: continue
    rows.append({"run_id": j["run_id"], "label": j["label"], "effort": j["effort"], "qid": j["qid"], "rep": j["rep"],
                 "qtype": j["qtype"], "judge": j["judge"], "pass_rate": np.mean(j["passes"]),
                 "fabrication": bool(j.get("fabrication")), "passes": tuple(j["passes"])})
jd = pd.DataFrame(rows)
tok = []
for p in (V2/"runs").glob("*.json"):
    r = json.loads(p.read_text())
    if "error" in r: continue
    tok.append({"label": r["label"], "effort": r["effort"], "in": r["in_tok"], "out": r["out_tok"],
                "reason": r.get("reason_tok") or 0, "cached": r.get("cached_tok") or 0, "or_cost": r.get("or_cost") or 0,
                "provider": r.get("provider")})
tok = pd.DataFrame(tok)
interp = jd[jd.qtype=="interpretive"]; calib = jd[jd.qtype=="deterministic"]
g = ["label","effort"]
summary = pd.DataFrame({
    "det_value":   det.groupby(g).value.mean(),
    "empty_rate":  det.groupby(g).empty.mean(),
    "int_astra":   interp[interp.judge=="astra"].groupby(g).pass_rate.mean(),
    "int_gemini":  interp[interp.judge=="gemini"].groupby(g).pass_rate.mean(),
    "fabrication": interp.groupby(g).fabrication.mean(),
    "reason_tok":  tok.groupby(g).reason.mean(),
    "cached_share": (tok.groupby(g).cached.sum() / tok.groupby(g)["in"].sum()),
    "or_cost_total": tok.groupby(g).or_cost.sum(),
    "provider":    tok.groupby(g).provider.agg(lambda s: s.mode().iloc[0] if len(s.dropna()) else None),
}).round(3)
summary["int_mean"] = summary[["int_astra","int_gemini"]].mean(axis=1).round(3)
summary["overall"]  = ((summary.det_value*15 + summary.int_mean*10)/25).round(3)
summary = summary.sort_values("overall", ascending=False)
summary.to_csv(V2/"results"/"summary.csv"); print(summary.to_string())
w = interp.pivot_table(index="run_id", columns="judge", values="passes", aggfunc="first").dropna()
if len(w):
    agree = np.mean([a==b for pa,pb in zip(w.astra, w.gemini) for a,b in zip(pa,pb)])
    print(f"\njudge-vs-judge criterion agreement (interpretive): {agree:.3f} over {len(w)} answers")
m = calib.merge(det[det.rep==1][["run_id","value"]], on="run_id"); m["j1"] = m.passes.apply(lambda t: t[0])
for jn, gg in m.groupby("judge"):
    print(f"judge {jn} vs python on det value: agree={np.mean(gg.j1==gg.value):.3f} (lenient={np.mean(gg.j1 & ~gg.value):.3f}, strict={np.mean(~gg.j1 & gg.value):.3f})")
print("\nwritten:", V2/"results"/"summary.csv")
